In [1]:
pip install -q langchain langchain-openai pypdf faiss-cpu tiktoken

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains.retrieval_qa import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# --- STEP 1: LOAD ---
loader = PyPDFLoader("HRPolicyManual2023.pdf")
data = loader.load()

In [ ]:
# --- STEP 2: CHUNK ---
# We use overlap to ensure context isn't lost at the cut-off points
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

In [ ]:
# --- STEP 3: STORE (Vector DB) ---
# This turns text into numbers (embeddings) and saves them locally
embeddings = OpenAIEmbeddings()
vector_db = FAISS.from_documents(chunks, embeddings)

In [ ]:
# --- STEP 4: RETRIEVE & CHAT ---
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # "Stuffs" all retrieved chunks into the prompt
    retriever=vector_db.as_retriever()
)

In [ ]:
# Example Query
query = "What is the main conclusion of this document?"
response = qa_chain.invoke(query)
print(response["result"])